## Installation
Gymnasium isn't part of the Python standard library, so it needs to be installed once per environment (computer / virtual environment / notebook server). If you've already installed it, you can skip this cell — running `pip install` again is harmless, it'll just confirm it's already there.


In [ ]:
%pip install "gymnasium[classic-control]" matplotlib

## Imports
Same idea as any Python script: we import the libraries we need before using them.

- `gymnasium` — the RL environment toolkit itself
- `numpy` — for arrays and numerical operations (Gymnasium observations come back as NumPy arrays)
- `matplotlib.pyplot` — for plotting our agent's learning progress, and for building our video clips


In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

SEED = 0
np.random.seed(SEED)

## Creating the Environment
We create environments with `gym.make(environment_id)`. The id is just a string Gymnasium looks up in its registry — `"Acrobot-v1"` here. (The `-v1` is a version number; environments occasionally get updated, and the version number tells you exactly which rules apply.)


In [10]:
env = gym.make("Acrobot-v1")
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<AcrobotEnv<Acrobot-v1>>>>>

In [6]:
observation, info = env.reset(seed=SEED)

print("Observation:", observation)
print("Info:", info)

Observation: [ 0.99962485  0.02738891  0.9989402  -0.04602639 -0.09180529 -0.09669447]
Info: {}


## 5. Understanding the Observation
The printed array has **6 numbers**. Acrobot's documentation tells us what they mean:

| Index | Meaning |
|---|---|
| 0 | cos(theta1) — cosine of the first joint's angle |
| 1 | sin(theta1) — sine of the first joint's angle |
| 2 | cos(theta2) — cosine of the second joint's angle |
| 3 | sin(theta2) — sine of the second joint's angle |
| 4 | angular velocity of joint 1 |
| 5 | angular velocity of joint 2 |

In [11]:
print("Observation Space:", env.observation_space)
print("Shape:", env.observation_space.shape)
print("Lower bounds:", env.observation_space.low)
print("Upper bounds:", env.observation_space.high)

Observation Space: Box([ -1.        -1.        -1.        -1.       -12.566371 -28.274334], [ 1.        1.        1.        1.       12.566371 28.274334], (6,), float32)
Shape: (6,)
Lower bounds: [ -1.        -1.        -1.        -1.       -12.566371 -28.274334]
Upper bounds: [ 1.        1.        1.        1.       12.566371 28.274334]


In [29]:
print("Action space:", env.action_space)
print("Number of actions:", env.action_space.n)

# .sample() picks a uniformly random valid action
print("A random action:", env.action_space.sample())

Action space: Discrete(3)
Number of actions: 3
A random action: 2


In [30]:
observation, info = env.reset(seed=SEED)

action = env.action_space.sample()  # pick a random action
observation, reward, terminated, truncated, info = env.step(action)

print("Action taken:", action)
print("New observation:", observation)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)

Action taken: 0
New observation: [ 0.9998245   0.01873245  0.995746   -0.09214022  0.00529764 -0.3585254 ]
Reward: -1.0
Terminated: False
Truncated: False


In [31]:
observation, info = env.reset(seed=SEED)
total_reward = 0
steps = 0

while True:
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    steps+= 1

    done = terminated or truncated
    if done:
        break

In [ ]:
from IPython.display import HTML
from matplotlib import animation


def collect_random_episode_frames(seed=SEED, max_steps=500):
    """Run one episode with random actions and return the list of rendered frames."""
    render_env = gym.make("Acrobot-v1", render_mode="rgb_array")
    obs, info = render_env.reset(seed=seed)

    frames = [render_env.render()]
    for _ in range(max_steps):
        action = render_env.action_space.sample()
        _, _, terminated, truncated, _ = render_env.step(action)
        frames.append(render_env.render())
        if terminated or truncated:
            break

    render_env.close()
    return frames

def make_animation(frames, title=""):
    """Turn a list of image frames into an inline, playable animation."""
    fig, ax = plt.subplots()
    ax.axis("off")
    if title:
        ax.set_title(title)
    img = ax.imshow(frames[0])

    def update(i):
        img.set_data(frames[i])
        return [img]

    anim = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=40, blit=True
    )
    plt.close(fig)  # prevents a duplicate static image from also being displayed
    return anim

random_frames = collect_random_episode_frames()
print(f"Collected {len(random_frames)} frames from the random-action episode.")

random_anim = make_animation(random_frames, title="Random actions")
HTML(random_anim.to_jshtml())
